In [2]:
import pandas as pd,  numpy as np

In [3]:
df = pd.read_csv("../data/raw/activity.csv")
df

,user_id,event_date,session_count,feature_used
0,1,2025-03-26,1,payment
1,1,2025-01-01,3,loan_apply
2,1,2025-03-18,4,payment
3,1,2025-03-06,1,promo_click
4,1,2025-02-01,2,promo_click
...,...,...,...,...
2954,250,2025-04-07,3,loan_apply
2955,250,2025-03-22,3,browse
2956,250,2025-04-08,3,promo_click
2957,250,2025-04-10,4,loan_apply


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2959 entries, 0 to 2958
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   user_id        2959 non-null   int64
 1   event_date     2959 non-null   str  
 2   session_count  2959 non-null   int64
 3   feature_used   2959 non-null   str  
dtypes: int64(2), str(2)
memory usage: 92.6 KB


### We wanna make recency column to make comparison and validation with days_since_last_activity column in status.csv

In [9]:
# Reference date for recency
reference_date = df["event_date"].max()
print(reference_date)

2025-04-30 00:00:00


In [7]:
df["event_date"] = pd.to_datetime(df["event_date"])

# Fetch last activity_date from each user
last_activity = df.groupby("user_id")["event_date"].max().reset_index(name="last_activity_date")

last_activity


,user_id,last_activity_date
0,1,2025-03-26
1,2,2025-04-04
2,3,2025-04-22
3,4,2025-04-25
4,5,2025-04-25
...,...,...
245,246,2025-04-14
246,247,2025-04-10
247,248,2025-04-28
248,249,2025-04-30


In [10]:
# Calculate recency each user
last_activity["recency_days"] = (
    reference_date - last_activity["last_activity_date"]
).dt.days

last_activity

,user_id,last_activity_date,recency_days
0,1,2025-03-26,35
1,2,2025-04-04,26
2,3,2025-04-22,8
3,4,2025-04-25,5
4,5,2025-04-25,5
...,...,...,...
245,246,2025-04-14,16
246,247,2025-04-10,20
247,248,2025-04-28,2
248,249,2025-04-30,0


In [12]:
status_df = pd.read_csv("../data/raw/status.csv")

comparison_df = status_df[
    ["user_id", "days_since_last_activity"]
].merge(
    last_activity,
    on="user_id",
    how="left",
    validate="one_to_one"
)

comparison_df

,user_id,days_since_last_activity,last_activity_date,recency_days
0,1,51,2025-03-26,35
1,2,39,2025-04-04,26
2,3,33,2025-04-22,8
3,4,33,2025-04-25,5
4,5,29,2025-04-25,5
...,...,...,...,...
245,246,10,2025-04-14,16
246,247,89,2025-04-10,20
247,248,77,2025-04-28,2
248,249,16,2025-04-30,0


Because almost (even all of that) days_since_last_activity not match with recency_days which is the definition of days_since_last_activity, then we drop column days_since_last_activity in status.csv


In [15]:
activity = df.copy()

print(f"number of duplicate values: {activity.duplicated().sum()}")
print(f"\nnumber of null values: \n{activity.isna().sum()}")

number of duplicate values: 7

number of null values: 
user_id          0
event_date       0
session_count    0
feature_used     0
dtype: int64


In [16]:
activity = activity.drop_duplicates()
activity["event_date"] = pd.to_datetime(activity["event_date"])
activity.info()

<class 'pandas.DataFrame'>
Index: 2952 entries, 0 to 2958
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   user_id        2952 non-null   int64         
 1   event_date     2952 non-null   datetime64[us]
 2   session_count  2952 non-null   int64         
 3   feature_used   2952 non-null   str           
dtypes: datetime64[us](1), int64(2), str(1)
memory usage: 115.3 KB


In [17]:
activity.to_csv("../data/clean/activity_clean.csv")